In [29]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [30]:
from catboost import CatBoostRegressor 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [31]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [32]:
encoder = LabelEncoder()
normalize = MinMaxScaler()

In [33]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=numpy.array(train[i[0]])
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

In [34]:
train = train.fillna(0)
test = test.fillna(0)

train_x = train.drop(columns=['efs'])
train_y = train['efs']

for i in test:
    test[i]=test[i].to_numpy()
train.head()

clf = LGBMRegressor(n_estimators=4200).fit(train_x, train_y)
model = SelectFromModel(clf, prefit=True)

weight = model.transform(train_x).reshape(-1)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012914 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 28800, number of used features: 57
[LightGBM] [Info] Start training from score 0.539306


In [35]:
lgbm_hyperparams={ 'colsample_bytree': 0.3394676841949491,
                   'drop_rate': 0.005229247069083676,
                   'learning_rate': 0.08939376710901481,
                   'max_bin': 3192,
                   'max_depth': 5655,
                   'max_drop': 3654,
                   'min_child_samples': 9011,
                   'min_data_in_leaf': 1320,
                   'n_estimators': 5619,
                   'num_leaves': 8002,
                   'objective': 'regression_l1',
                   'reg_alpha': 0.3092258349709437,
                   'reg_lambda': 0.5130123398875309,
                   'skip_drop': 0.5672900003846416,
                   'verbosity': -1}

XGB_hyperparams = {'colsample_bytree': 0.6679438268550072,
                   'gamma': 2,
                   'learning_rate': 0.19750405984350627,
                   'max_depth': 20,
                   'min_child_weight': 9,
                   'n_estimators': 2338,
                   'objective': 'binary:logistic',
                   'reg_alpha': 44,
                   'reg_lambda': 0.5688906203980144,
                   'subsample': 0.5716652066840949}

RandomForest = {'criterion': 'poisson',
                'max_depth': 7,
                'max_features': 'sqrt',
                'max_leaf_nodes': 5,
                'min_samples_leaf': 8,
                'min_samples_split': 19,
                'min_weight_fraction_leaf': 0.01391485916360663,
                'random_state': 90}
Fold=5

In [36]:
kf = KFold(n_splits=Fold)

LGBM_pre_train = numpy.zeros(len(train))
LGBM_pre_test = numpy.zeros(len(test))

xgb_pre_train = numpy.zeros(len(train))
xgb_pre_test = numpy.zeros(len(test))

cat_pre_train = numpy.zeros(len(train))
cat_pre_test = numpy.zeros(len(test))

RandomF_pre_train = numpy.zeros(len(train))
RandomF_pre_test = numpy.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    weight_ = weight[train_index[0]: train_index[len(train_index)-1]]

    lightgbm = LGBMRegressor()
    xgboost = XGBRegressor()
    catboost = CatBoostRegressor(learning_rate=0.009,
                                 depth=5,
                                 l2_leaf_reg=3.5,
                                 min_child_samples=32,
                                 grow_policy='Depthwise',
                                 iterations=1000,
                                 eval_metric='RMSE',
                                 od_type='Iter',
                                 od_wait=50,
                                 random_state=42,
                                 logging_level='Silent')
    RandomF = GradientBoostingRegressor()
    
    catboost.fit(x_fold, y_fold, sample_weight=weight_)
    xgboost.fit(x_fold, y_fold, sample_weight=weight_)
    lightgbm.fit(x_fold, y_fold, eval_sample_weight=weight_)
    RandomF.fit(x_fold, y_fold, weight_)

    LGBM_pre_train+=lightgbm.predict(train_x)
    LGBM_pre_test+=lightgbm.predict(test)

    xgb_pre_train+=xgboost.predict(train_x)
    xgb_pre_test+=xgboost.predict(test)

    cat_pre_train+=catboost.predict(train_x)
    cat_pre_test+=catboost.predict(test)

    RandomF_pre_train += RandomF.predict(train_x)
    RandomF_pre_test += RandomF.predict(test)

    print(RandomF.score(x_test_fold, y_test_fold))
    print(catboost.score(x_test_fold, y_test_fold))
    print(xgboost.score(x_test_fold, y_test_fold))
    print(lightgbm.score(x_test_fold, y_test_fold))

LGBM_pre_train = LGBM_pre_train/Fold
LGBM_pre_test = LGBM_pre_test/Fold

FOLD: 0
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010212 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 23039, number of used features: 57
[LightGBM] [Info] Start training from score 0.539780
0.14831425027727607
0.1372707004586592
0.02415448430561451
0.19019719132665247
FOLD: 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013425 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 28799, number of used features: 57
[LightGBM] [Info] Start training from score 0.539324
0.15807701212095537
0.13206313170132689
0.21662148508099954
0.293603336478618
FOLD: 2
[LightGBM

In [41]:
new_train_data = pandas.DataFrame({'RandomF' : RandomF_pre_train,
                             'LGBM' : LGBM_pre_train,
                             'xgb' : xgb_pre_train,
                             'cat' : cat_pre_train})

new_test_data = pandas.DataFrame({'RandomF' : RandomF_pre_test,
                             'LGBM' : LGBM_pre_test,
                             'xgb' : xgb_pre_test,
                             'cat' : cat_pre_test})
model=GradientBoostingRegressor()
model.fit(new_train_data, train_y)

GradientBoostingRegressor()

In [50]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = ((RandomF_pre_test/Fold)+(LGBM_pre_test/Fold)+(xgb_pre_test/Fold)+(cat_pre_test/Fold))/4
#model.predict(new_test_data)
#((RandomF_pre_test/10)+(LGBM_pre_test/5)+(xgb_pre_test/5)+(cat_pre_test/5))/4
test_predictions

array([0.31971783, 0.35968502, 0.27761703])

In [46]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.319718
1,28801,0.359685
2,28802,0.277617


In [40]:
submission.to_csv('submission.csv', index = False)